<a href="https://colab.research.google.com/github/dystaSatria/Deep-Learning/blob/main/Internship%20Projects/specificity/specificity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import OrderedDict

class DenseLayer(nn.Module):
    """DenseNet'in temel yapı taşı - Tek bir yoğun katman"""

    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super(DenseLayer, self).__init__()

        # Bottleneck tasarımı: 1x1 conv -> 3x3 conv
        self.add_module('norm1', nn.BatchNorm2d(num_input_features))
        self.add_module('relu1', nn.ReLU(inplace=True))
        self.add_module('conv1', nn.Conv2d(num_input_features, bn_size * growth_rate,
                                          kernel_size=1, stride=1, bias=False))

        self.add_module('norm2', nn.BatchNorm2d(bn_size * growth_rate))
        self.add_module('relu2', nn.ReLU(inplace=True))
        self.add_module('conv2', nn.Conv2d(bn_size * growth_rate, growth_rate,
                                          kernel_size=3, stride=1, padding=1, bias=False))

        self.drop_rate = drop_rate

    def forward(self, x):
        # DenseNet'in özgün özelliği: önceki tüm katmanların çıktılarını birleştirme
        if isinstance(x, torch.Tensor):
            prev_features = [x]
        else:
            prev_features = x

        concated_features = torch.cat(prev_features, 1)

        # Bottleneck işlemi
        bottleneck_output = self.conv1(self.relu1(self.norm1(concated_features)))
        new_features = self.conv2(self.relu2(self.norm2(bottleneck_output)))

        # Dropout uygulaması
        if self.drop_rate > 0:
            new_features = F.dropout(new_features, p=self.drop_rate, training=self.training)

        return new_features


class DenseBlock(nn.Module):
    """Yoğun blok: Birden fazla yoğun katmanın birleşimi"""

    def __init__(self, num_layers, num_input_features, bn_size, growth_rate, drop_rate):
        super(DenseBlock, self).__init__()

        for i in range(num_layers):
            layer = DenseLayer(
                num_input_features + i * growth_rate,
                growth_rate=growth_rate,
                bn_size=bn_size,
                drop_rate=drop_rate
            )
            self.add_module('denselayer%d' % (i + 1), layer)

    def forward(self, init_features):
        features = [init_features]

        for name, layer in self.named_children():
            new_features = layer(features)
            features.append(new_features)

        # Tüm özellik haritalarını birleştir (DenseNet'in ana özelliği)
        return torch.cat(features, 1)


class Transition(nn.Module):
    """Geçiş katmanı: Boyut azaltma ve özellik sıkıştırma"""

    def __init__(self, num_input_features, num_output_features):
        super(Transition, self).__init__()

        self.add_module('norm', nn.BatchNorm2d(num_input_features))
        self.add_module('relu', nn.ReLU(inplace=True))
        self.add_module('conv', nn.Conv2d(num_input_features, num_output_features,
                                          kernel_size=1, stride=1, bias=False))
        self.add_module('pool', nn.AvgPool2d(kernel_size=2, stride=2))

    def forward(self, x):
        return self.pool(self.conv(self.relu(self.norm(x))))


class DenseNet(nn.Module):
    """
    DenseNet mimarisi

    Args:
        growth_rate (int): Her katmanın eklediği kanal sayısı (k)
        block_config (list): Her blokta kaç katman olduğunu belirten liste
        num_init_features (int): İlk konvolüsyon katmanının çıktı kanalları
        bn_size (int): Bottleneck boyutu çarpanı
        drop_rate (float): Dropout oranı
        num_classes (int): Sınıf sayısı
    """

    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0, num_classes=1000):

        super(DenseNet, self).__init__()

        # İlk konvolüsyon katmanı
        self.features = nn.Sequential(OrderedDict([
            ('conv0', nn.Conv2d(3, num_init_features, kernel_size=7, stride=2,
                               padding=3, bias=False)),
            ('norm0', nn.BatchNorm2d(num_init_features)),
            ('relu0', nn.ReLU(inplace=True)),
            ('pool0', nn.MaxPool2d(kernel_size=3, stride=2, padding=1)),
        ]))

        # Dense bloklar ve geçiş katmanları
        num_features = num_init_features
        for i, num_layers in enumerate(block_config):
            # Dense blok ekleme
            block = DenseBlock(
                num_layers=num_layers,
                num_input_features=num_features,
                bn_size=bn_size,
                growth_rate=growth_rate,
                drop_rate=drop_rate
            )
            self.features.add_module('denseblock%d' % (i + 1), block)
            num_features = num_features + num_layers * growth_rate

            # Son blok hariç geçiş katmanı ekleme
            if i != len(block_config) - 1:
                trans = Transition(num_input_features=num_features,
                                 num_output_features=num_features // 2)
                self.features.add_module('transition%d' % (i + 1), trans)
                num_features = num_features // 2

        # Son batch normalization
        self.features.add_module('norm5', nn.BatchNorm2d(num_features))

        # Sınıflandırıcı
        self.classifier = nn.Linear(num_features, num_classes)

        # Ağırlık başlatma
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1))
        out = torch.flatten(out, 1)
        out = self.classifier(out)
        return out


# Popüler DenseNet varyantları
def densenet121(num_classes=1000):
    """DenseNet-121 modeli"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 24, 16),
                   num_init_features=64, num_classes=num_classes)

def densenet169(num_classes=1000):
    """DenseNet-169 modeli"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 32, 32),
                   num_init_features=64, num_classes=num_classes)

def densenet201(num_classes=1000):
    """DenseNet-201 modeli"""
    return DenseNet(growth_rate=32, block_config=(6, 12, 48, 32),
                   num_init_features=64, num_classes=num_classes)


# Kullanım örneği
if __name__ == "__main__":
    # Model oluşturma
    model = densenet121(num_classes=10)  # CIFAR-10 için

    # Örnek veri
    x = torch.randn(4, 3, 224, 224)  # Batch size: 4, RGB, 224x224

    # Forward pass
    output = model(x)
    print(f"Giriş boyutu: {x.shape}")
    print(f"Çıkış boyutu: {output.shape}")

    # Model parametreleri
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Toplam parametre sayısı: {total_params:,}")
    print(f"Eğitilebilir parametre sayısı: {trainable_params:,}")

Giriş boyutu: torch.Size([4, 3, 224, 224])
Çıkış boyutu: torch.Size([4, 10])
Toplam parametre sayısı: 6,964,106
Eğitilebilir parametre sayısı: 6,964,106
